# 3 - External validation

Tests the signature (Table 2, from notebook 2) on GSE267650, a mouse series that took no
part in its construction (Section 4.6): non-operated controls, ischemia without
reperfusion, and reperfused animals at six timepoints. Two comparisons are scored:
reperfused vs controls (`GSE267650`) and reperfused vs ischemia-only (`GSE267650direct`).

| Output | Use |
|---|---|
| `results/HOLDOUT_DE_GSE267650*.csv`, `results/HOLDOUT_design_GSE267650.csv` | differential expression (DESeq2) and design |
| `results/validation_holdout_GSE267650*.csv` | per-gene comparison and median oriented log2 fold change |
| `results/validation_holdout_perm_GSE267650*.csv` | concordance and permutation test (limma; the row `perm = 0` holds the observed values) |

In [1]:
# Move to the project root (the folder that contains run_all.ps1), so that relative
# paths work whether the notebook is started from notebook/ or from the root.
root <- normalizePath(getwd(), winslash = "/", mustWork = TRUE)
while (!file.exists(file.path(root, "run_all.ps1")) && dirname(root) != root) {
  root <- dirname(root)
}
if (!file.exists(file.path(root, "run_all.ps1"))) {
  stop("project root not found: no run_all.ps1 above ", getwd())
}
setwd(root)
.libPaths(c(file.path(root, "env", "rlib"), .libPaths()))
Sys.setenv(RENAL_IRI_ROOT = root)
options(stringsAsFactors = FALSE, warn = 1)
cat(R.version.string, "\nroot: ", root, "\n", sep = "")

R version 4.4.3 (2025-02-28)
root: /Users/robertocasale/Desktop/lavoro_FBF/renal_shield_perfusion


## Setup

In [2]:
source("R/helpers.R")

[2026-10-07 17:02:28] INFO  helpers.R loaded; constants and robust_fetch() ready.



## 1. Differential expression in GSE267650

DESeq2 with group as the only term (`R/validation/holdout_load_GSE267650.R`).

In [3]:
source("R/validation/holdout_load_GSE267650.R")

[2026-10-07 17:02:37] INFO  GSE267650: 21017 genes x 57 samples



[2026-10-07 17:02:38] INFO  dropped 16 Azin1 samples (genotype confound)



Warning message in eval(quote(list(...)), env):
“NAs introduced by coercion”


[2026-10-07 17:02:38] INFO  control n=5 | ischemia-only n=5 | reperfusion arms: WT_IRI_4h, WT_IRI_16h, WT_IRI_24h, WT_IRI_36h, WT_IRI_48h, WT_IRI_72h



[2026-10-07 17:02:38] INFO  after collapsing duplicate symbols: 21017 genes



[2026-10-07 17:02:38] INFO  design written: control=5 | ischemia_only=5 | reperfusion=31 | excluded_Azin1=16



converting counts to integer mode



[2026-10-07 17:02:58] INFO  reperfused vs non-operated controls             5 vs 31 -> results/HOLDOUT_DE_GSE267650.csv (16533 genes, 5432 flagged)



converting counts to integer mode



[2026-10-07 17:03:17] INFO  reperfused vs ischemia-only                     5 vs 31 -> 16525 genes, 5348 flagged



## 2. Direction of the signature genes

Concordance and median oriented log2 fold change, for each comparison
(`R/validation/holdout_replication.R`).

In [4]:
for (HOLDOUT_ACC in c("GSE267650", "GSE267650direct")) {
  source("R/validation/holdout_replication.R")
}

[2026-10-07 17:03:24] INFO    Mus musculus: 451 outdated rodent symbols updated before ortholog mapping



GSE267650: 256 of 256 signature genes measured | concordant 250 (97.7%) | median oriented log2 fold change +1.68


[2026-10-07 17:03:37] INFO  WROTE results/validation_holdout_GSE267650.csv



[2026-10-07 17:03:45] INFO    Mus musculus: 450 outdated rodent symbols updated before ortholog mapping



GSE267650direct: 256 of 256 signature genes measured | concordant 243 (94.9%) | median oriented log2 fold change +1.30


[2026-10-07 17:03:55] INFO  WROTE results/validation_holdout_GSE267650direct.csv



## 3. Permutation test

The mean oriented moderated t of the signature genes, compared with 1,000 random
reassignments of the group labels (`R/validation/holdout_permutation_unpaired.R`).

In [5]:
for (HOLDOUT_TAG in c("GSE267650", "GSE267650direct")) {
  HOLDOUT_B <- 1000
  source("R/validation/holdout_permutation_unpaired.R")
}

[2026-10-07 17:03:56] INFO  GSE267650: 5 vs 31, 16536 genes, B = 1000 sampled labellings



[2026-10-07 17:04:02] INFO    Mus musculus: 451 outdated rodent symbols updated before ortholog mapping



[2026-10-07 17:04:11] INFO  signature genes traceable to a row: 256 of 256



[2026-10-07 17:04:11] INFO  permutation group = 3.77e+05 labellings -> SAMPLING B = 1000



   200 / 1000 (30 s)
   400 / 1000 (58 s)
   600 / 1000 (97 s)
   800 / 1000 (133 s)
   1000 / 1000 (165 s)

==== label permutation (GSE267650, B = 1000) ====
  conc    observed    0.973 | null median   0.447, max   0.992 | >= observed:   72/1000 | p = 0.0729
  mean_t  observed    6.600 | null median  -0.076, max   3.149 | >= observed:    0/1000 | p = 0.0010  (the smallest attainable p)


[2026-10-07 17:06:57] INFO  GSE267650direct: 5 vs 31, 16525 genes, B = 1000 sampled labellings



[2026-10-07 17:07:04] INFO    Mus musculus: 450 outdated rodent symbols updated before ortholog mapping



[2026-10-07 17:07:15] INFO  signature genes traceable to a row: 256 of 256



[2026-10-07 17:07:15] INFO  permutation group = 3.77e+05 labellings -> SAMPLING B = 1000



   200 / 1000 (24 s)
   400 / 1000 (49 s)
   600 / 1000 (73 s)
   800 / 1000 (97 s)
   1000 / 1000 (121 s)

==== label permutation (GSE267650direct, B = 1000) ====
  conc    observed    0.941 | null median   0.449, max   0.988 | >= observed:  115/1000 | p = 0.1159
  mean_t  observed    5.669 | null median  -0.071, max   3.006 | >= observed:    0/1000 | p = 0.0010  (the smallest attainable p)
